<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_GeoPolitical_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

# MOS :: Middle East Oil Security
### An LLM-Based Geopolitical What-If Simulator

> **Action → Consequence → Reaction → Consequence → ...**

**MOS** explores how geopolitical situations may evolve when strategic actors respond to one another in an uncertain and changing world.

**Chanakya** proposes strategic alternatives.  
**Centaur** evaluates their relative likelihood.  
**Shakuni** -- the dice roller -- selects a path under uncertainty.  
**ZeitWorld** translates the selected Event into the next WorldState.

The cycle repeats, creating one possible **Scenario** from an initial hypothetical Trigger.

> **The objective is not to predict the future, but to explore plausible, path-dependent futures arising from strategic choices.**

In [1]:
from IPython.display import Markdown, display
from datetime import datetime
import pytz
print('ॐ श्री सरस्वत्यै नमः',datetime.now(pytz.timezone('Asia/Calcutta')))

ॐ श्री सरस्वत्यै नमः 2026-10-04 09:22:55.495010+05:30


In [2]:
!pip install -q python-docx
#from docx.oxml import OxmlElement
#from docx.oxml.ns import qn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.5 MB/s eta 0:00:00


In [3]:
!wget -O CentaurLogo.png -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/logo-S.png

!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py
!wget -O mosCentaur.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosCentaur.py
!wget -O mosReporter.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosReporter.py
!wget -O mosDOCx.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosDOCx.py

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *
from mosDOCx import *

In [4]:
import importlib

import mos
import mosZeitWorld
import mosChanakya
import mosCentaur
import mosReporter
import mosDOCx

importlib.reload(mos)
importlib.reload(mosZeitWorld)
importlib.reload(mosChanakya)
importlib.reload(mosCentaur)
importlib.reload(mosReporter)
importlib.reload(mosDOCx)

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *

## Initialise Runtime Environment

Authenticate OpenAI, select the working LLM, mount Google Drive and define the paths to the MOS data, roles and prompts.

The directory listings provide a quick check that all required resources are available.

In [5]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

# Real-world evidence, LLM roles and prompts
# are stored persistently in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"
CT_PATH = f"{MOS_HOME}/Centaur"

#!ls -l "{WORLDFACTS_DIR}"
#!ls -lh "{WORLDFACTS_DIR}/MD"
#!ls -lh "{WORLDFACTS_DIR}/NEWS"
#!ls -lh "{ZW_PATH}"
#!ls -lh "{CH_PATH}"
#!ls -lh "{CT_PATH}"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com
Mounted at /content/drive


## Get World Facts

In [6]:
# Build WorldFactBase
#
# Static PDF-derived Markdown files and live Moneycontrol news
# are assembled into a single corpus used to initialise W0.
# No real-world information enters the simulation after W0.
# ------------------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)

Static sources : 8
Live sources   : 1
Characters     : 235,072
Words          : 35,600


### Defining the World

In [7]:
# Define the WorldState variables -- SET A
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------

worldVariables_SetA = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [8]:
# Define the WorldState variables  -- SET B
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------
worldVariables_SetB = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction=(
            "Higher index means higher oil price."
        )
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction=(
            "Higher index means greater oil flow."
        )
    ),

    WorldVariable(
        name="Regional Diplomatic Engagement",
        description=(
            "Degree of active diplomatic engagement aimed at managing, "
            "containing or resolving the regional crisis, including "
            "direct negotiations, mediation, backchannels and "
            "de-escalation mechanisms."
        ),
        direction=(
            "Higher index means greater diplomatic engagement "
            "and stronger functioning channels for de-escalation."
        )
    ),

    WorldVariable(
        name="Economic Pressure on Iran",
        description=(
            "Degree of economic pressure on Iran arising from sanctions, "
            "trade and financial restrictions, blockade measures, "
            "export constraints and related economic coercion."
        ),
        direction=(
            "Higher index means greater economic pressure on Iran."
        )
    ),

    WorldVariable(
        name="US/Regional Partner Cohesion",
        description=(
            "Degree of alignment between the United States and key "
            "regional partners, including Israel, Saudi Arabia and "
            "other Gulf states, in responding to Iran and the "
            "regional crisis."
        ),
        direction=(
            "Higher index means greater alignment and willingness "
            "to coordinate with US policy."
        )
    ),

    WorldVariable(
        name="Iran Domestic Stability",
        description=(
            "Degree of stability and resilience within Iran's political, "
            "military, security and domestic system under the pressures "
            "generated by the regional crisis."
        ),
        direction=(
            "Higher index means greater Iranian domestic stability."
        )
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction=(
            "Higher index means greater US-Iran tension."
        )
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )

]

In [9]:
worldVariables = worldVariables_SetB

## Agents and their Roles

### ZeitWorld

In [10]:
# Load ZeitWorld roles and prompts
#
# ZW_Role                  : defines ZeitWorld's role and discipline
# ZW_Initialisation_Prompt : creates W0 from the WorldFactBase
# ZW_Update_Prompt         : evolves the current WorldState after an Event
# ------------------------------------------------------------

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")
ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [11]:
# Instantiate ZeitWorld
#
# ZeitWorld is created with its LLM client, model and governing role.
# The initialised flag confirms whether W0 has already been created.
# ------------------------------------------------------------

ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
#print(ZeitWorld.initialised)
#print(ZeitWorld.model)

### Chanakya

In [12]:
# Load Chanakya roles and strategy
#
# Generic Role : defines the common strategic discipline of all Chanakyas
# US Role      : defines US objectives, interests and constraints
# Iran Role    : defines Iranian objectives, interests and constraints
# BinaryHD     : generates two contrasting Hawk and Dove candidates
# ============================================================

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")
CH_BinaryHD     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")

In [13]:
# Instantiate Chanakya agents
#
# Each Chanakya represents a national actor with its own
# interests and constraints, using the current Binary Hawk/Dove strategy.
# The registry maps each actor to its corresponding Chanakya.
# ============================================================

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_BinaryHD
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_BinaryHD
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


### Centaur

In [14]:
# Load Centaur roles
#
# Generic Role : defines the common adjudication discipline of all Centaurs
# UNSG Role    : provides a fair, independent UN diplomatic perspective
# ============================================================

CT_Generic_Role = getPrompt(f"{CT_PATH}/CT_Generic_Role_v0.txt")
CT_UNSG_Role      = getPrompt(f"{CT_PATH}/CT_UNSG_Role_v0.txt")

In [15]:
# Instantiate Centaur adjudicator
#
# CentaurUNSG evaluates Chanakya's Candidate Events from the
# perspective of a fair and independent UN Secretary-General.
# ============================================================
Centaur_UNSG = Centaur_Agent(
    agent_id="CentaurUNSG",
    generic_role=CT_Generic_Role,
    adjudication_role=CT_UNSG_Role
)

## Initialisation of Scenario

In [16]:
# Initialise Simulation State
#
# W stores the evolving WorldStates of the simulation.
# E stores the sequence of Events defining the Scenario.
# W0_Template defines the empty structure from which W0 is created.
# ------------------------------------------------------------

E = []
W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

# Initialise the first WorldState
#
# ZeitWorld uses the WorldFactBase and W0 template to create W0.
# The completed WorldState is automatically appended to W as W[0].
# ============================================================

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

#print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 57,108 | Output: 1,929 | Reasoning: 187 | Total: 59,037


In [17]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

In [18]:
# Define initial Trigger options
#
# Two alternative Events are available to start the simulation.
# One is manually selected as E0; thereafter Events are generated
# and selected by the simulation.
# ============================================================

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for US"
)


In [19]:
# Select the initial Trigger
#
# The manually selected DefaultTrigger starts the first epoch.
# Its selection weight is None because it was not chosen by Shakuni.
# ============================================================
DefaultTrigger = Trigger1
#DefaultTrigger = Trigger2
DefaultTrigger.show()
print(DefaultTrigger.selection_weight)


Event     : 
Actor     : Iran
Action    : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
Rationale : Initial Trigger for Iran
selection_weight : None
None


In [20]:
# Initialise / Reset Scenario
#
# Retains W0 and clears all subsequent WorldStates and Events,
# restoring the simulation to its starting condition.
# ============================================================

def resetWorld():
    del W[1:]
    E.clear()

    print("Scenario reset to W0.")

# ------------------------------------------------------------

resetWorld()

Scenario reset to W0.


### Define Single Epoch

In [21]:
def runEpoch():
    # ============================================================
    # EPOCH STEP 1 — Establish Current Trigger
    #
    # The epoch is derived from the current WorldState history.
    # For the first epoch, DefaultTrigger becomes E0.
    # Thereafter, the last Reaction becomes the next Trigger.
    # ============================================================

    epoch = len(W) - 1
    if len(E) == 0:
        DefaultTrigger.event_id = f"E{epoch}"
        E.append(DefaultTrigger)
    else:
        assert E[epoch] is E[-1]

    #E[epoch].show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 1 complete" )

    # ============================================================
    # EPOCH STEP 2 — Update WorldState
    #
    # ZeitWorld applies Trigger E[epoch] to W[epoch] and creates
    # the next WorldState W[epoch+1].
    # ============================================================

    _ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

    #E[epoch].show()
    #W[epoch+1].show(-2)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 2 complete" )

    # ============================================================
    # EPOCH STEP 3 — Select Active Actor
    #
    # The Turn Controller identifies which actor responds to the
    # Current Trigger and activates its corresponding Chanakya.
    # ============================================================

    #ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)
    ActiveChanakya = getActiveChanakya(E[epoch], ChanakyaRegistry)
    #ActiveChanakya.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 3 complete" )

    # ============================================================
    # EPOCH STEP 4 — Generate Candidate Reactions
    #
    # The Active Chanakya evaluates the updated WorldState and
    # generates Candidate Events as possible reactions to the Trigger.
    # ============================================================

    Candidates = ActiveChanakya.propose(
        client=client,
        model=model,
        trigger=E[epoch],
        world=W[epoch+1]
    )

    #for i, candidate in enumerate(Candidates):
    #    print(f"\nCandidate Reaction {i+1}")
    #    print("-" * 60)
    #    candidate.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 4 complete" )

    # ============================================================
    # EPOCH STEP 5 — Adjudicate Candidate Reactions
    #
    # Centaur evaluates the Candidate Events against the current
    # WorldState and assigns each a relative simulation weight.
    # ============================================================

    weights = Centaur_UNSG.evaluate(
        client,
        model,
        E[epoch],
        W[epoch + 1],
        Candidates
    )

    #print("\nCentaur Weights")
    #print("----------------")
    print("Weights",weights)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 5 complete" )

    # ============================================================
    #
    # Centaur weights represent the relative conditional likelihood
    # of each Candidate at this decision point.
    #
    # Shakuni makes a single stochastic selection using these weights.
    # The selected Candidate becomes the Reaction, retains its Centaur
    # weight as selection_weight, receives the next Event ID, and is
    # appended to E as the Trigger for the next epoch.
    # ============================================================

    Reaction = Shakuni(Candidates, weights)

    Reaction.event_id = f"E{len(E)}"
    E.append(Reaction)

    #print("\nShakuni has spoken")
    #print("------------------")
    #Reaction.show()
    #print(Reaction.selection_weight)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 6 complete" )
    print("Epoch ends ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")



##Scenario | Multiple Epochs

In [22]:
MaxEvents = 4
_scenarioID = "TR1ME4"

resetWorld()

while len(E) < MaxEvents:
    runEpoch()

# Apply the final Event and close the Scenario
_ = ZeitWorld.update(W, E[-1], ZW_Update_Prompt)

# Structural integrity check
assert len(E) == MaxEvents
assert len(W) == MaxEvents + 1
assert len(W) == len(E) + 1

print("\nScenario complete")
print("-----------------")
print("Events      :", len(E))
print("WorldStates :", len(W))

Scenario reset to W0.
epoch : 0 len(W): 1 len(E): 1  ***** Step # 1 complete
Updating WorldState ---------- W0
Model: gpt-5.6-luna | Effort: medium | Input: 3,059 | Output: 2,405 | Reasoning: 305 | Total: 5,464
Appended  WorldState ---------- W1
epoch : 0 len(W): 2 len(E): 1  ***** Step # 2 complete
epoch : 0 len(W): 2 len(E): 1  ***** Step # 3 complete
Chanakya working ---------- ChanakyaUS_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,273 | Output: 314 | Reasoning: 102 | Total: 2,587
epoch : 0 len(W): 2 len(E): 1  ***** Step # 4 complete
Centaur working ---------- CentaurUNSG
Model: gpt-5.6-luna | Effort: medium | Input: 2,111 | Output: 114 | Reasoning: 98 | Total: 2,225
Weights [0.72, 0.28]
epoch : 0 len(W): 2 len(E): 1  ***** Step # 5 complete
epoch : 0 len(W): 2 len(E): 2  ***** Step # 6 complete
Epoch ends ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
epoch : 1 len(W): 2 len(E): 2  ***** Step # 1 complete
Updating WorldState ---------- W1
Model: gpt-5

## Reporter

In [23]:
MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
#WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
#ZW_PATH = f"{MOS_HOME}/ZeitWorld"
#CH_PATH = f"{MOS_HOME}/Chanakya"
#CT_PATH = f"{MOS_HOME}/Centaur"
PR_PATH = f"{MOS_HOME}/Reporter"

#!ls -l "{WORLDFACTS_DIR}"
#!ls -lh "{WORLDFACTS_DIR}/MD"
#!ls -lh "{WORLDFACTS_DIR}/NEWS"
#!ls -lh "{ZW_PATH}"
#!ls -lh "{CH_PATH}"
#!ls -lh "{CT_PATH}"
#!ls -lh "{PR_PATH}"

In [24]:
# ============================================================
# Load PressReporter Roles and Prompts
#
# Generic Role defines reporting discipline.
# Narrative Prompt produces the human-readable Scenario history.
# Structured Prompt produces the machine-oriented Markdown record.
# ============================================================

PR_Generic_Role = getPrompt(f"{PR_PATH}/PR_Generic_Role_v0.txt")

#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_v0.txt")
#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v2.txt")
PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v3.txt")

PR_Structured   = getPrompt(f"{PR_PATH}/PR_Structured_v0.txt")


# ============================================================
# Instantiate Human Reporter
# ============================================================

Reporter = Reporter_Agent(
    client=client,
    model=model,
    agent_id="Vyasa",
    generic_role=PR_Generic_Role
)

In [25]:
#ScenarioText = PressReporter._scenarioText(W, E)
#print(ScenarioText)

In [26]:
# ============================================================
# Generate Narrative Scenario Report
#
# PressReporter converts the completed Scenario (W + E) into a
# human-readable narrative using the Narrative reporting prompt.
# The resulting report is then formatted and saved as a Word document.
# ============================================================
NarrativeReport = Reporter.GenerateReport(
    W,
    E,
    PR_Narrative
)

#print(NarrativeReport)

NarrativeFile = createNarrativeDocx(
    NarrativeReport,
    W,
    E,
    scenarioID = _scenarioID,
    title="Middle East Oil Security — Scenario Brief"
)

Reporter working ---------- Vyasa | Narrative
Model: gpt-5.6-luna | Effort: medium | Input: 11,191 | Output: 981 | Reasoning: 198 | Total: 12,172
Word report created: /content/MOS_TR1ME4_261004_092541.docx


In [27]:
# ============================================================
# Generate Structured Scenario Report
#
# Produces a structured Markdown representation of the completed
# Scenario for subsequent machine-based analysis.
#
# ============================================================
'''
StructuredReport = Reporter.GenerateReport(
    W,
    E,
    PR_Structured
)

display(Markdown(StructuredReport))
'''

Reporter working ---------- Vyasa | Narrative
Model: gpt-5.6-luna | Effort: medium | Input: 10,866 | Output: 9,328 | Reasoning: 131 | Total: 20,194


# Scenario Summary

- **Number of Events:** 4
- **Number of WorldStates:** 5
- **Initial WorldState:** W0
- **Terminal WorldState:** W4

## Event E0

- **Event ID:** E0
- **Actor:** Iran
- **Action:** Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
- **Rationale:** Initial Trigger for Iran
- **Selection Weight:** None
- **Source WorldState:** W0
- **Resulting WorldState:** W1

### Resulting WorldState: W1

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 112
- **Change from W0:** 100 → 112
- **Status:** Brent prices are expected to rise further from already elevated and volatile levels because Iran has declared the Strait of Hormuz closed, is preventing tanker transit and is attacking US naval forces. Alternative routes and limited remaining flows constrain but do not eliminate the supply shock.
- **Rationale:** The index rises from 100 to 112 because the Event further restricts the already materially reduced Middle Eastern oil flows and increases the risk to remaining shipments. Existing inventory depletion and shipping disruptions justify a significant but not extreme increase because bypass routes and alternative arrangements remain operative.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 68
- **Change from W0:** 100 → 68
- **Status:** Commercial oil and petroleum-product transit through the Strait of Hormuz is at a further reduced level following Iran's declared closure and orders to prevent tanker movement. Only limited flows may continue through bypasses, escorted movements or other exceptional arrangements.
- **Rationale:** The index falls from 100 to 68 because the Event directly targets all commercial tanker transit, intensifying the pre-existing severe constraints. The decline is substantial but not to zero because the current state records alternative pipelines, overland routes, ship-to-ship transfers and military escorts that can partially sustain movement.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 92
- **Change from W0:** 100 → 92
- **Status:** Diplomatic engagement is further weakened by Iran's declared closure of the Strait of Hormuz and attacks on US naval forces. Existing mediation and indirect contacts remain possible, but the immediate escalation makes de-escalation channels less effective.
- **Rationale:** The index falls from 100 to 92 because the Event adds a direct maritime and military escalation while existing contacts are already stalled. The decrease is moderate rather than severe because the current conditions still preserve possible indirect contacts and regional mediation.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 100
- **Change from W0:** 100 → 100
- **Status:** Economic pressure on Iran remains extensive and increasing as previously described. The Event changes maritime risk and military confrontation but does not itself impose a new external sanctions or blockade measure on Iran.
- **Rationale:** The index remains at 100 because the Event is an Iranian military and maritime action, not a new US or other external economic restriction. Existing sanctions, blockade measures and export constraints remain operative without a sufficiently direct additional economic-pressure change specified by the Event.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 105
- **Change from W0:** 100 → 105
- **Status:** Iran's closure of the Strait of Hormuz and attacks on US naval forces increase the immediate shared maritime-security threat and strengthen the basis for US coordination with affected regional partners. Existing concerns about retaliation and dependence on US protection remain.
- **Rationale:** The index rises from 100 to 105 because the Event creates a more direct common threat to US and regional maritime and energy interests, reinforcing incentives for coordination. The increase is limited because the current state records qualified alignment and continuing partner concerns about escalation.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 100
- **Change from W0:** 100 → 100
- **Status:** Iran's governing system remains in power and its security forces retain the ability to suppress organized domestic opposition. Existing strains from leadership transition, repression, inflation, energy shortages and wartime damage remain; the Event does not establish a distinct domestic political or security change.
- **Rationale:** The index remains at 100 because the Event concerns external military operations and does not directly report a change in regime control, elite cohesion, repression capacity or domestic opposition. Existing domestic-stability conditions therefore remain operative without a justified numerical adjustment.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 115
- **Change from W0:** 100 → 115
- **Status:** US-Iran tension is further escalated by Iran's formal closure of the Strait of Hormuz and direct attacks on US naval forces. Military confrontation now includes an explicit challenge to US maritime operations in the Gulf alongside the existing blockade, sanctions and stalled negotiations.
- **Rationale:** The index rises from 100 to 115 because the Event adds direct attacks on US forces and a declared attempt to halt commercial transit, substantially intensifying the existing intermittent confrontation. The increase is large but not maximal because the current state already records very high tension and ongoing military, economic and diplomatic hostility.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 112
- **Change from W0:** 100 → 112
- **Status:** Armed conflict is more intense in the Gulf following Iran's attacks on US naval forces and its orders to prevent tanker transit. The existing conflict across the Gulf, Lebanon, Iraq and Yemen continues, with the maritime confrontation now directly expanded around commercial shipping and US military operations.
- **Rationale:** The index rises from 100 to 112 because the Event introduces direct Iranian attacks on US naval forces and escalates the ongoing contest over commercial maritime transit. The increase is significant because kinetic activity and operational risk intensify in an already geographically broad conflict, while the existing theaters remain otherwise unchanged.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.

## Event E1

- **Event ID:** E1
- **Actor:** US
- **Action:** Launch a coordinated military campaign with regional partners to suppress Iranian naval, missile and coastal capabilities threatening US forces and tanker transit, while expanding escorted shipping operations through the Strait of Hormuz.
- **Rationale:** Iran has directly attacked US naval forces and is attempting to close a vital maritime chokepoint. Imposing immediate military costs and demonstrating the capability and willingness to restore navigation could strengthen deterrence, protect US forces and reassure affected partners.
- **Selection Weight:** 0.72
- **Source WorldState:** W1
- **Resulting WorldState:** W2

### Resulting WorldState: W2

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 110
- **Change from W1:** 112 → 110
- **Status:** Brent prices remain substantially elevated and volatile. Expanded escorted shipping and efforts to suppress Iranian threats reduce some immediate transit risk, but the US campaign and continuing Iranian resistance sustain a significant risk premium and ongoing supply uncertainty.
- **Rationale:** The index falls from 112 to 110 because expanded escorts and attacks on capabilities threatening tanker transit modestly improve the security of some shipments. The decline is limited because the campaign introduces additional military risk and the underlying disruption, depleted inventories and Iranian closure remain operative.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.
  - The United States and regional partners are conducting military operations and escorted shipping to reduce threats to tanker transit.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 74
- **Change from W1:** 68 → 74
- **Status:** Oil transit through the Strait of Hormuz increases from its previous severely reduced level as US and regional forces expand escorted shipping operations and suppress threats to tanker movement. Transit remains well below baseline because Iran's closure order, attacks and the continuing military confrontation constrain commercial flows.
- **Rationale:** The index rises from 68 to 74 because the Event directly expands escorted shipping and targets Iranian naval, missile and coastal capabilities threatening tanker transit. The increase is moderate rather than restorative because Iranian forces continue to contest passage, the declared closure remains operative and the broader conflict still limits reliable commercial movement.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.
  - The United States and regional partners have expanded escorted shipping operations through the Strait of Hormuz.
  - US and regional military operations are targeting Iranian naval, missile and coastal capabilities that threaten tanker transit.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 87
- **Change from W1:** 92 → 87
- **Status:** Diplomatic engagement is further weakened as the United States launches a coordinated military campaign with regional partners. Existing mediation and indirect contacts remain possible, but expanded combat operations reduce the immediate effectiveness of de-escalation channels.
- **Rationale:** The index falls from 92 to 87 because the Event expands direct military operations and formalizes broader operational coordination among the United States and regional partners while prior contacts are already stalled. The decline is moderate because the current conditions still preserve possible indirect contacts and regional mediation.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.
  - The United States and regional partners are conducting an expanded coordinated military campaign against Iranian capabilities.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 104
- **Change from W1:** 100 → 104
- **Status:** Economic pressure on Iran increases as the coordinated campaign reinforces the existing naval blockade and constrains Iranian maritime, coastal and export-related capabilities. No new sanctions measure is specified, so the increase remains limited.
- **Rationale:** The index rises from 100 to 104 because the Event adds military enforcement against capabilities supporting Iranian control of maritime access and tanker disruption, intensifying the operative blockade and export constraints. The change is modest because the Event does not itself announce new financial sanctions, trade restrictions or a separate economic measure.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.
  - US and regional military operations are further constraining Iranian naval, missile and coastal capabilities relevant to maritime access and oil exports.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 111
- **Change from W1:** 105 → 111
- **Status:** US and regional partner cohesion increases as the United States and partners conduct a coordinated campaign and expand escorted shipping operations. The alignment remains qualified by the continuing risks and costs of broader military operations.
- **Rationale:** The index rises from 105 to 111 because the Event demonstrates active operational coordination among the United States and regional partners against a shared maritime and energy threat. The increase is significant but limited because the current state records continuing partner concerns about escalation and dependence on US protection.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.
  - The United States and regional partners are conducting a coordinated military campaign and expanded escorted shipping operations.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 96
- **Change from W1:** 100 → 96
- **Status:** Iran's governing system remains in power, but the coordinated US and regional campaign places additional pressure on Iranian military and coastal capabilities and adds to existing wartime and economic strains. No collapse of regime control or security-force capacity is established.
- **Rationale:** The index falls from 100 to 96 because expanded attacks on Iranian naval, missile and coastal capabilities increase military pressure and compound existing inflation, energy shortages, reduced oil revenue and wartime damage. The decrease is limited because the current state still records regime control and continuing security-force capacity, with no distinct domestic political breakdown reported.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.
  - US and regional military operations are imposing additional pressure on Iranian naval, missile and coastal capabilities.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 122
- **Change from W1:** 115 → 122
- **Status:** US-Iran tension rises further as the United States launches a coordinated campaign against Iranian military capabilities and expands escorted shipping through the contested Strait of Hormuz. Direct military confrontation now includes sustained US operations against Iranian coastal and maritime threats.
- **Rationale:** The index rises from 115 to 122 because the Event expands the scope and coordination of direct US military action against Iran while maintaining the existing blockade, sanctions and maritime confrontation. The increase is substantial because the campaign deepens active hostilities, but it is not maximal because the current state already records very high tension and ongoing conflict.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.
  - The United States has launched a coordinated military campaign with regional partners against Iranian naval, missile and coastal capabilities.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 121
- **Change from W1:** 112 → 121
- **Status:** Regional armed conflict intensifies as the United States and regional partners conduct coordinated strikes against Iranian naval, missile and coastal capabilities while expanding escorted shipping operations. Existing fighting across the Gulf, Lebanon, Iraq and Yemen continues, with the Gulf theater becoming more operationally active.
- **Rationale:** The index rises from 112 to 121 because the Event adds a coordinated military campaign and sustained operations around tanker transit to an already active, geographically broad conflict. The increase is large because kinetic activity and operational scope expand directly, while the index remains below an extreme level because the Event does not establish new combat theaters beyond the existing regional conflict.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.
  - The United States and regional partners are conducting coordinated military operations against Iranian naval, missile and coastal capabilities.

## Event E2

- **Event ID:** E2
- **Actor:** Iran
- **Action:** Intensify a dispersed regional coercion campaign by expanding missile, drone and asymmetric maritime attacks against US and partner military assets, supporting Iran-linked operations across existing theaters, and maintaining the enforced closure of the Strait of Hormuz.
- **Rationale:** Iran could seek to impose higher operational and political costs on the coordinated campaign, demonstrate that suppression of its capabilities will not produce compliance, and strengthen its bargaining leverage by widening the risks faced by the United States and its regional partners.
- **Selection Weight:** 0.75
- **Source WorldState:** W2
- **Resulting WorldState:** W3

### Resulting WorldState: W3

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 116
- **Change from W2:** 110 → 116
- **Status:** Brent prices rise as Iran intensifies attacks against military assets and maintains the enforced closure of the Strait of Hormuz. Expanded coercion increases perceived disruption and security risk, while existing military operations, depleted inventories and reduced production continue to support a substantial risk premium.
- **Rationale:** The index rises from 110 to 116 because intensified missile, drone and asymmetric maritime attacks increase the immediate risk to regional security and oil transit, while the enforced Hormuz closure remains operative. The increase is significant but limited because escorted shipping, military efforts to suppress threats and existing alternative routes continue to support some movement.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.
  - The United States and regional partners are conducting military operations and escorted shipping to reduce threats to tanker transit.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and regional partner military assets.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 65
- **Change from W2:** 74 → 65
- **Status:** Oil transit through the Strait of Hormuz declines as Iran intensifies attacks and maintains the enforced closure. Escorts and suppression operations continue to enable limited movement, but expanded asymmetric threats reduce the reliability and volume of commercial transit.
- **Rationale:** The index falls from 74 to 65 because intensified maritime, missile and drone attacks directly increase the danger to tanker movement while Iran continues enforcing the closure. The decline is substantial but not a return to the earlier lowest level because military escorts, alternative routes and ongoing suppression operations still sustain partial flows.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.
  - The United States and regional partners have expanded escorted shipping operations through the Strait of Hormuz.
  - US and regional military operations are targeting Iranian naval, missile and coastal capabilities that threaten tanker transit.
  - Iran has intensified dispersed missile, drone and asymmetric maritime attacks against US and partner military assets while maintaining the enforced closure.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 81
- **Change from W2:** 87 → 81
- **Status:** Diplomatic engagement weakens further as Iran expands attacks across existing theaters and maintains the enforced closure of the Strait of Hormuz. Existing mediation and indirect contacts remain possible but are increasingly obstructed by active coercion and military escalation.
- **Rationale:** The index falls from 87 to 81 because the Event expands Iran's military coercion against the United States and regional partners while prior contacts are already stalled. The decline is moderate because the current state still records possible indirect contacts and regional mediation, even though their immediate effectiveness is reduced.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.
  - The United States and regional partners are conducting an expanded coordinated military campaign against Iranian capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks across existing theaters and continues enforcing the Strait of Hormuz closure.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 106
- **Change from W2:** 104 → 106
- **Status:** Economic pressure on Iran increases modestly as intensified attacks and continued closure prompt further military enforcement of the existing blockade and constrain Iranian maritime and export-related activity. No new sanctions or financial restrictions are specified.
- **Rationale:** The index rises from 104 to 106 because Iran's intensified attacks and continued closure give the existing blockade and export constraints greater operational effect, increasing pressure on maritime access and oil-related activity. The change remains small because the Event does not itself add new sanctions, trade restrictions or financial measures.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.
  - US and regional military operations are further constraining Iranian naval, missile and coastal capabilities relevant to maritime access and oil exports.
  - Iran's intensified attacks and enforced Strait of Hormuz closure are increasing the operational constraints on Iranian maritime and export activity.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 115
- **Change from W2:** 111 → 115
- **Status:** US and regional partner cohesion increases as Iran expands attacks against their military assets and operations across existing theaters. The shared threat supports continued coordination, although the broader conflict continues to impose risks and costs on partners.
- **Rationale:** The index rises from 111 to 115 because the Event broadens the immediate common threat to the United States and regional partners through intensified missile, drone and asymmetric attacks and continued disruption of Hormuz transit. The increase is limited because existing conditions already record active coordination and continuing partner concerns about escalation.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.
  - The United States and regional partners are conducting a coordinated military campaign and expanded escorted shipping operations.
  - Iran has expanded attacks against US and partner military assets and supports Iran-linked operations across existing theaters.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 92
- **Change from W2:** 96 → 92
- **Status:** Iran's governing system remains in power but faces additional military and economic strain as the coercion campaign expands and hostile operations continue. The state retains security-force capacity, and no domestic political collapse is established.
- **Rationale:** The index falls from 96 to 92 because intensifying attacks and the continued closure deepen Iran's exposure to military pressure, while existing inflation, energy shortages, reduced oil revenue and wartime damage remain operative. The decline is limited because the current state still records regime control and continuing security-force capacity without a distinct domestic breakdown.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.
  - US and regional military operations are imposing additional pressure on Iranian naval, missile and coastal capabilities.
  - Iran's expanded regional coercion campaign increases military, economic and organizational strain on the state.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 128
- **Change from W2:** 122 → 128
- **Status:** US-Iran tension rises as Iran expands missile, drone and asymmetric maritime attacks against US and partner military assets while maintaining the enforced closure of the Strait of Hormuz. Direct confrontation becomes more geographically dispersed across the existing conflict theaters.
- **Rationale:** The index rises from 122 to 128 because the Event intensifies direct attacks on US and partner military assets and sustains the central maritime confrontation despite the existing US campaign and blockade. The increase is substantial because active hostilities broaden in tempo and scope, but it remains below an extreme ceiling because the Event does not establish a wholly new theater.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.
  - The United States has launched a coordinated military campaign with regional partners against Iranian naval, missile and coastal capabilities.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 129
- **Change from W2:** 121 → 129
- **Status:** Regional armed conflict intensifies and becomes more dispersed as Iran expands missile, drone and asymmetric maritime attacks, supports Iran-linked operations across existing theaters and maintains the enforced Hormuz closure. Existing fighting across the Gulf, Lebanon, Iraq and Yemen continues with increased operational pressure.
- **Rationale:** The index rises from 121 to 129 because the Event directly increases the intensity and geographic breadth of kinetic activity across the existing Gulf, Lebanon, Iraq and Yemen theaters and sustains confrontation around tanker transit. The increase is large because both attack tempo and dispersed operational activity expand, while no entirely new theater is established.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.
  - The United States and regional partners are conducting coordinated military operations against Iranian naval, missile and coastal capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.

## Event E3

- **Event ID:** E3
- **Actor:** US
- **Action:** Expand the coordinated military campaign by intensifying strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities, while tightening enforcement of the naval blockade and increasing escorted shipping operations through the Strait of Hormuz.
- **Rationale:** Iran's expanded attacks directly threaten US forces, regional partners and freedom of navigation. A more forceful response could impose additional costs, restore deterrence, protect maritime operations and demonstrate that continued coercion will not compel US withdrawal.
- **Selection Weight:** 0.8
- **Source WorldState:** W3
- **Resulting WorldState:** W4

### Resulting WorldState: W4

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 117
- **Change from W3:** 116 → 117
- **Status:** Brent prices remain elevated as the United States intensifies strikes on Iranian capabilities and tightens enforcement of the Strait of Hormuz blockade. Increased escorted shipping provides some countervailing support for transit, but expanded military operations sustain a substantial disruption and security risk premium.
- **Rationale:** The index rises from 116 to 117 because intensified US strikes and tighter blockade enforcement increase immediate regional escalation and disruption risks. The increase is limited because expanded escorted shipping operations provide some reassurance regarding tanker movement and partially offset the added risk.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.
  - The United States and regional partners are conducting military operations and escorted shipping to reduce threats to tanker transit.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and regional partner military assets.
  - The United States has intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities and tightened blockade enforcement.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 69
- **Change from W3:** 65 → 69
- **Status:** Oil transit through the Strait of Hormuz increases modestly as the United States expands escorted shipping operations, although the enforced closure, tighter blockade and intensified military activity continue to constrain reliability and volume.
- **Rationale:** The index rises from 65 to 69 because increased escorted shipping directly enables more tanker movement through the Strait despite continuing Iranian attacks and the declared closure. The increase is limited because tighter blockade enforcement and intensified strikes maintain a high-risk operating environment and do not restore normal transit.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.
  - The United States and regional partners have expanded escorted shipping operations through the Strait of Hormuz.
  - US and regional military operations are targeting Iranian naval, missile and coastal capabilities that threaten tanker transit.
  - Iran has intensified dispersed missile, drone and asymmetric maritime attacks against US and partner military assets while maintaining the enforced closure.
  - The United States has tightened blockade enforcement and increased escorted shipping operations through the Strait.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 78
- **Change from W3:** 81 → 78
- **Status:** Diplomatic engagement weakens as the United States expands military strikes and blockade enforcement while Iran continues attacks and the enforced Strait of Hormuz closure. Existing mediation and indirect contacts remain possible but are further obstructed by active escalation.
- **Rationale:** The index falls from 81 to 78 because the Event expands direct US military action and tightens coercive measures while existing contacts are already stalled. The decline is moderate because the current state still records possible indirect contacts and regional mediation, but their immediate operating space is reduced.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.
  - The United States and regional partners are conducting an expanded coordinated military campaign against Iranian capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks across existing theaters and continues enforcing the Strait of Hormuz closure.
  - The United States has intensified strikes and tightened blockade enforcement, further narrowing near-term de-escalation space.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 110
- **Change from W3:** 106 → 110
- **Status:** Economic pressure on Iran increases as the United States tightens enforcement of the naval blockade and intensifies strikes on capabilities relevant to maritime access and exports. No additional sanctions or financial restrictions are specified.
- **Rationale:** The index rises from 106 to 110 because tighter blockade enforcement directly increases constraints on Iranian maritime access and export-related activity, while strikes on coastal and naval capabilities reinforce those constraints. The increase is moderate because the Event does not add new sanctions, trade restrictions or financial measures.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.
  - US and regional military operations are further constraining Iranian naval, missile and coastal capabilities relevant to maritime access and oil exports.
  - Iran's intensified attacks and enforced Strait of Hormuz closure are increasing the operational constraints on Iranian maritime and export activity.
  - The United States has tightened enforcement of the naval blockade and intensified strikes on Iranian capabilities supporting maritime and export activity.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 118
- **Change from W3:** 115 → 118
- **Status:** US and regional partner cohesion increases as the coordinated campaign expands and escorted shipping operations are increased in response to the shared threat from Iran. Existing partner concerns about escalation and conflict costs remain.
- **Rationale:** The index rises from 115 to 118 because the Event expands coordinated military operations and maritime protection involving the United States and regional partners. The increase is moderate rather than large because existing conditions already record active cooperation and continuing partner concerns about escalation.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.
  - The United States and regional partners are conducting a coordinated military campaign and expanded escorted shipping operations.
  - Iran has expanded attacks against US and partner military assets and supports Iran-linked operations across existing theaters.
  - The United States has intensified the coordinated campaign and increased escorted shipping operations with regional partners.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 88
- **Change from W3:** 92 → 88
- **Status:** Iran's governing system remains in power but faces greater military, economic and organizational strain as US strikes intensify and blockade enforcement tightens. Security-force capacity remains present, and no domestic political collapse is established.
- **Rationale:** The index falls from 92 to 88 because intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities and tighter blockade enforcement increase pressure on the state. The decline is substantial but limited because the current state still records regime control and continuing security-force capacity without a distinct domestic breakdown.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.
  - US and regional military operations are imposing additional pressure on Iranian naval, missile and coastal capabilities.
  - Iran's expanded regional coercion campaign increases military, economic and organizational strain on the state.
  - The United States has intensified strikes on Iranian command-and-control and military capabilities and tightened blockade enforcement, increasing pressure on state operations.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 134
- **Change from W3:** 128 → 134
- **Status:** US-Iran tension rises sharply as the United States intensifies strikes on Iranian military and command capabilities, tightens the naval blockade and expands escorted shipping while Iran maintains attacks and the enforced Strait of Hormuz closure.
- **Rationale:** The index rises from 128 to 134 because the Event directly expands US military action, blockade enforcement and the maritime confrontation with Iran. The increase is substantial given the escalation across several operational domains, while it remains below an extreme ceiling because the Event continues the existing conflict rather than establishing a wholly new theater.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.
  - The United States has launched a coordinated military campaign with regional partners against Iranian naval, missile and coastal capabilities.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.
  - The United States has intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities while tightening blockade enforcement.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 135
- **Change from W3:** 129 → 135
- **Status:** Regional armed conflict intensifies as the United States expands strikes across Iranian military and command capabilities, tightens the blockade and increases escorted shipping operations while Iran continues attacks across existing theaters and maintains the enforced Hormuz closure.
- **Rationale:** The index rises from 129 to 135 because the Event increases the intensity of direct kinetic operations against Iranian missile, drone, naval, coastal and command-and-control capabilities and sustains combat around tanker transit. The increase is substantial because the operational tempo expands across multiple existing conflict domains, without establishing an entirely new theater.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.
  - The United States and regional partners are conducting coordinated military operations against Iranian naval, missile and coastal capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.
  - The United States has intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities and increased military activity around the Strait.

In [28]:


display(Markdown(StructuredReport))

# Scenario Summary

- **Number of Events:** 4
- **Number of WorldStates:** 5
- **Initial WorldState:** W0
- **Terminal WorldState:** W4

## Event E0

- **Event ID:** E0
- **Actor:** Iran
- **Action:** Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
- **Rationale:** Initial Trigger for Iran
- **Selection Weight:** None
- **Source WorldState:** W0
- **Resulting WorldState:** W1

### Resulting WorldState: W1

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 112
- **Change from W0:** 100 → 112
- **Status:** Brent prices are expected to rise further from already elevated and volatile levels because Iran has declared the Strait of Hormuz closed, is preventing tanker transit and is attacking US naval forces. Alternative routes and limited remaining flows constrain but do not eliminate the supply shock.
- **Rationale:** The index rises from 100 to 112 because the Event further restricts the already materially reduced Middle Eastern oil flows and increases the risk to remaining shipments. Existing inventory depletion and shipping disruptions justify a significant but not extreme increase because bypass routes and alternative arrangements remain operative.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 68
- **Change from W0:** 100 → 68
- **Status:** Commercial oil and petroleum-product transit through the Strait of Hormuz is at a further reduced level following Iran's declared closure and orders to prevent tanker movement. Only limited flows may continue through bypasses, escorted movements or other exceptional arrangements.
- **Rationale:** The index falls from 100 to 68 because the Event directly targets all commercial tanker transit, intensifying the pre-existing severe constraints. The decline is substantial but not to zero because the current state records alternative pipelines, overland routes, ship-to-ship transfers and military escorts that can partially sustain movement.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 92
- **Change from W0:** 100 → 92
- **Status:** Diplomatic engagement is further weakened by Iran's declared closure of the Strait of Hormuz and attacks on US naval forces. Existing mediation and indirect contacts remain possible, but the immediate escalation makes de-escalation channels less effective.
- **Rationale:** The index falls from 100 to 92 because the Event adds a direct maritime and military escalation while existing contacts are already stalled. The decrease is moderate rather than severe because the current conditions still preserve possible indirect contacts and regional mediation.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 100
- **Change from W0:** 100 → 100
- **Status:** Economic pressure on Iran remains extensive and increasing as previously described. The Event changes maritime risk and military confrontation but does not itself impose a new external sanctions or blockade measure on Iran.
- **Rationale:** The index remains at 100 because the Event is an Iranian military and maritime action, not a new US or other external economic restriction. Existing sanctions, blockade measures and export constraints remain operative without a sufficiently direct additional economic-pressure change specified by the Event.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 105
- **Change from W0:** 100 → 105
- **Status:** Iran's closure of the Strait of Hormuz and attacks on US naval forces increase the immediate shared maritime-security threat and strengthen the basis for US coordination with affected regional partners. Existing concerns about retaliation and dependence on US protection remain.
- **Rationale:** The index rises from 100 to 105 because the Event creates a more direct common threat to US and regional maritime and energy interests, reinforcing incentives for coordination. The increase is limited because the current state records qualified alignment and continuing partner concerns about escalation.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 100
- **Change from W0:** 100 → 100
- **Status:** Iran's governing system remains in power and its security forces retain the ability to suppress organized domestic opposition. Existing strains from leadership transition, repression, inflation, energy shortages and wartime damage remain; the Event does not establish a distinct domestic political or security change.
- **Rationale:** The index remains at 100 because the Event concerns external military operations and does not directly report a change in regime control, elite cohesion, repression capacity or domestic opposition. Existing domestic-stability conditions therefore remain operative without a justified numerical adjustment.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 115
- **Change from W0:** 100 → 115
- **Status:** US-Iran tension is further escalated by Iran's formal closure of the Strait of Hormuz and direct attacks on US naval forces. Military confrontation now includes an explicit challenge to US maritime operations in the Gulf alongside the existing blockade, sanctions and stalled negotiations.
- **Rationale:** The index rises from 100 to 115 because the Event adds direct attacks on US forces and a declared attempt to halt commercial transit, substantially intensifying the existing intermittent confrontation. The increase is large but not maximal because the current state already records very high tension and ongoing military, economic and diplomatic hostility.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 112
- **Change from W0:** 100 → 112
- **Status:** Armed conflict is more intense in the Gulf following Iran's attacks on US naval forces and its orders to prevent tanker transit. The existing conflict across the Gulf, Lebanon, Iraq and Yemen continues, with the maritime confrontation now directly expanded around commercial shipping and US military operations.
- **Rationale:** The index rises from 100 to 112 because the Event introduces direct Iranian attacks on US naval forces and escalates the ongoing contest over commercial maritime transit. The increase is significant because kinetic activity and operational risk intensify in an already geographically broad conflict, while the existing theaters remain otherwise unchanged.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.

## Event E1

- **Event ID:** E1
- **Actor:** US
- **Action:** Launch a coordinated military campaign with regional partners to suppress Iranian naval, missile and coastal capabilities threatening US forces and tanker transit, while expanding escorted shipping operations through the Strait of Hormuz.
- **Rationale:** Iran has directly attacked US naval forces and is attempting to close a vital maritime chokepoint. Imposing immediate military costs and demonstrating the capability and willingness to restore navigation could strengthen deterrence, protect US forces and reassure affected partners.
- **Selection Weight:** 0.72
- **Source WorldState:** W1
- **Resulting WorldState:** W2

### Resulting WorldState: W2

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 110
- **Change from W1:** 112 → 110
- **Status:** Brent prices remain substantially elevated and volatile. Expanded escorted shipping and efforts to suppress Iranian threats reduce some immediate transit risk, but the US campaign and continuing Iranian resistance sustain a significant risk premium and ongoing supply uncertainty.
- **Rationale:** The index falls from 112 to 110 because expanded escorts and attacks on capabilities threatening tanker transit modestly improve the security of some shipments. The decline is limited because the campaign introduces additional military risk and the underlying disruption, depleted inventories and Iranian closure remain operative.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.
  - The United States and regional partners are conducting military operations and escorted shipping to reduce threats to tanker transit.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 74
- **Change from W1:** 68 → 74
- **Status:** Oil transit through the Strait of Hormuz increases from its previous severely reduced level as US and regional forces expand escorted shipping operations and suppress threats to tanker movement. Transit remains well below baseline because Iran's closure order, attacks and the continuing military confrontation constrain commercial flows.
- **Rationale:** The index rises from 68 to 74 because the Event directly expands escorted shipping and targets Iranian naval, missile and coastal capabilities threatening tanker transit. The increase is moderate rather than restorative because Iranian forces continue to contest passage, the declared closure remains operative and the broader conflict still limits reliable commercial movement.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.
  - The United States and regional partners have expanded escorted shipping operations through the Strait of Hormuz.
  - US and regional military operations are targeting Iranian naval, missile and coastal capabilities that threaten tanker transit.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 87
- **Change from W1:** 92 → 87
- **Status:** Diplomatic engagement is further weakened as the United States launches a coordinated military campaign with regional partners. Existing mediation and indirect contacts remain possible, but expanded combat operations reduce the immediate effectiveness of de-escalation channels.
- **Rationale:** The index falls from 92 to 87 because the Event expands direct military operations and formalizes broader operational coordination among the United States and regional partners while prior contacts are already stalled. The decline is moderate because the current conditions still preserve possible indirect contacts and regional mediation.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.
  - The United States and regional partners are conducting an expanded coordinated military campaign against Iranian capabilities.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 104
- **Change from W1:** 100 → 104
- **Status:** Economic pressure on Iran increases as the coordinated campaign reinforces the existing naval blockade and constrains Iranian maritime, coastal and export-related capabilities. No new sanctions measure is specified, so the increase remains limited.
- **Rationale:** The index rises from 100 to 104 because the Event adds military enforcement against capabilities supporting Iranian control of maritime access and tanker disruption, intensifying the operative blockade and export constraints. The change is modest because the Event does not itself announce new financial sanctions, trade restrictions or a separate economic measure.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.
  - US and regional military operations are further constraining Iranian naval, missile and coastal capabilities relevant to maritime access and oil exports.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 111
- **Change from W1:** 105 → 111
- **Status:** US and regional partner cohesion increases as the United States and partners conduct a coordinated campaign and expand escorted shipping operations. The alignment remains qualified by the continuing risks and costs of broader military operations.
- **Rationale:** The index rises from 105 to 111 because the Event demonstrates active operational coordination among the United States and regional partners against a shared maritime and energy threat. The increase is significant but limited because the current state records continuing partner concerns about escalation and dependence on US protection.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.
  - The United States and regional partners are conducting a coordinated military campaign and expanded escorted shipping operations.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 96
- **Change from W1:** 100 → 96
- **Status:** Iran's governing system remains in power, but the coordinated US and regional campaign places additional pressure on Iranian military and coastal capabilities and adds to existing wartime and economic strains. No collapse of regime control or security-force capacity is established.
- **Rationale:** The index falls from 100 to 96 because expanded attacks on Iranian naval, missile and coastal capabilities increase military pressure and compound existing inflation, energy shortages, reduced oil revenue and wartime damage. The decrease is limited because the current state still records regime control and continuing security-force capacity, with no distinct domestic political breakdown reported.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.
  - US and regional military operations are imposing additional pressure on Iranian naval, missile and coastal capabilities.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 122
- **Change from W1:** 115 → 122
- **Status:** US-Iran tension rises further as the United States launches a coordinated campaign against Iranian military capabilities and expands escorted shipping through the contested Strait of Hormuz. Direct military confrontation now includes sustained US operations against Iranian coastal and maritime threats.
- **Rationale:** The index rises from 115 to 122 because the Event expands the scope and coordination of direct US military action against Iran while maintaining the existing blockade, sanctions and maritime confrontation. The increase is substantial because the campaign deepens active hostilities, but it is not maximal because the current state already records very high tension and ongoing conflict.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.
  - The United States has launched a coordinated military campaign with regional partners against Iranian naval, missile and coastal capabilities.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 121
- **Change from W1:** 112 → 121
- **Status:** Regional armed conflict intensifies as the United States and regional partners conduct coordinated strikes against Iranian naval, missile and coastal capabilities while expanding escorted shipping operations. Existing fighting across the Gulf, Lebanon, Iraq and Yemen continues, with the Gulf theater becoming more operationally active.
- **Rationale:** The index rises from 112 to 121 because the Event adds a coordinated military campaign and sustained operations around tanker transit to an already active, geographically broad conflict. The increase is large because kinetic activity and operational scope expand directly, while the index remains below an extreme level because the Event does not establish new combat theaters beyond the existing regional conflict.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.
  - The United States and regional partners are conducting coordinated military operations against Iranian naval, missile and coastal capabilities.

## Event E2

- **Event ID:** E2
- **Actor:** Iran
- **Action:** Intensify a dispersed regional coercion campaign by expanding missile, drone and asymmetric maritime attacks against US and partner military assets, supporting Iran-linked operations across existing theaters, and maintaining the enforced closure of the Strait of Hormuz.
- **Rationale:** Iran could seek to impose higher operational and political costs on the coordinated campaign, demonstrate that suppression of its capabilities will not produce compliance, and strengthen its bargaining leverage by widening the risks faced by the United States and its regional partners.
- **Selection Weight:** 0.75
- **Source WorldState:** W2
- **Resulting WorldState:** W3

### Resulting WorldState: W3

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 116
- **Change from W2:** 110 → 116
- **Status:** Brent prices rise as Iran intensifies attacks against military assets and maintains the enforced closure of the Strait of Hormuz. Expanded coercion increases perceived disruption and security risk, while existing military operations, depleted inventories and reduced production continue to support a substantial risk premium.
- **Rationale:** The index rises from 110 to 116 because intensified missile, drone and asymmetric maritime attacks increase the immediate risk to regional security and oil transit, while the enforced Hormuz closure remains operative. The increase is significant but limited because escorted shipping, military efforts to suppress threats and existing alternative routes continue to support some movement.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.
  - The United States and regional partners are conducting military operations and escorted shipping to reduce threats to tanker transit.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and regional partner military assets.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 65
- **Change from W2:** 74 → 65
- **Status:** Oil transit through the Strait of Hormuz declines as Iran intensifies attacks and maintains the enforced closure. Escorts and suppression operations continue to enable limited movement, but expanded asymmetric threats reduce the reliability and volume of commercial transit.
- **Rationale:** The index falls from 74 to 65 because intensified maritime, missile and drone attacks directly increase the danger to tanker movement while Iran continues enforcing the closure. The decline is substantial but not a return to the earlier lowest level because military escorts, alternative routes and ongoing suppression operations still sustain partial flows.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.
  - The United States and regional partners have expanded escorted shipping operations through the Strait of Hormuz.
  - US and regional military operations are targeting Iranian naval, missile and coastal capabilities that threaten tanker transit.
  - Iran has intensified dispersed missile, drone and asymmetric maritime attacks against US and partner military assets while maintaining the enforced closure.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 81
- **Change from W2:** 87 → 81
- **Status:** Diplomatic engagement weakens further as Iran expands attacks across existing theaters and maintains the enforced closure of the Strait of Hormuz. Existing mediation and indirect contacts remain possible but are increasingly obstructed by active coercion and military escalation.
- **Rationale:** The index falls from 87 to 81 because the Event expands Iran's military coercion against the United States and regional partners while prior contacts are already stalled. The decline is moderate because the current state still records possible indirect contacts and regional mediation, even though their immediate effectiveness is reduced.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.
  - The United States and regional partners are conducting an expanded coordinated military campaign against Iranian capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks across existing theaters and continues enforcing the Strait of Hormuz closure.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 106
- **Change from W2:** 104 → 106
- **Status:** Economic pressure on Iran increases modestly as intensified attacks and continued closure prompt further military enforcement of the existing blockade and constrain Iranian maritime and export-related activity. No new sanctions or financial restrictions are specified.
- **Rationale:** The index rises from 104 to 106 because Iran's intensified attacks and continued closure give the existing blockade and export constraints greater operational effect, increasing pressure on maritime access and oil-related activity. The change remains small because the Event does not itself add new sanctions, trade restrictions or financial measures.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.
  - US and regional military operations are further constraining Iranian naval, missile and coastal capabilities relevant to maritime access and oil exports.
  - Iran's intensified attacks and enforced Strait of Hormuz closure are increasing the operational constraints on Iranian maritime and export activity.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 115
- **Change from W2:** 111 → 115
- **Status:** US and regional partner cohesion increases as Iran expands attacks against their military assets and operations across existing theaters. The shared threat supports continued coordination, although the broader conflict continues to impose risks and costs on partners.
- **Rationale:** The index rises from 111 to 115 because the Event broadens the immediate common threat to the United States and regional partners through intensified missile, drone and asymmetric attacks and continued disruption of Hormuz transit. The increase is limited because existing conditions already record active coordination and continuing partner concerns about escalation.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.
  - The United States and regional partners are conducting a coordinated military campaign and expanded escorted shipping operations.
  - Iran has expanded attacks against US and partner military assets and supports Iran-linked operations across existing theaters.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 92
- **Change from W2:** 96 → 92
- **Status:** Iran's governing system remains in power but faces additional military and economic strain as the coercion campaign expands and hostile operations continue. The state retains security-force capacity, and no domestic political collapse is established.
- **Rationale:** The index falls from 96 to 92 because intensifying attacks and the continued closure deepen Iran's exposure to military pressure, while existing inflation, energy shortages, reduced oil revenue and wartime damage remain operative. The decline is limited because the current state still records regime control and continuing security-force capacity without a distinct domestic breakdown.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.
  - US and regional military operations are imposing additional pressure on Iranian naval, missile and coastal capabilities.
  - Iran's expanded regional coercion campaign increases military, economic and organizational strain on the state.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 128
- **Change from W2:** 122 → 128
- **Status:** US-Iran tension rises as Iran expands missile, drone and asymmetric maritime attacks against US and partner military assets while maintaining the enforced closure of the Strait of Hormuz. Direct confrontation becomes more geographically dispersed across the existing conflict theaters.
- **Rationale:** The index rises from 122 to 128 because the Event intensifies direct attacks on US and partner military assets and sustains the central maritime confrontation despite the existing US campaign and blockade. The increase is substantial because active hostilities broaden in tempo and scope, but it remains below an extreme ceiling because the Event does not establish a wholly new theater.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.
  - The United States has launched a coordinated military campaign with regional partners against Iranian naval, missile and coastal capabilities.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 129
- **Change from W2:** 121 → 129
- **Status:** Regional armed conflict intensifies and becomes more dispersed as Iran expands missile, drone and asymmetric maritime attacks, supports Iran-linked operations across existing theaters and maintains the enforced Hormuz closure. Existing fighting across the Gulf, Lebanon, Iraq and Yemen continues with increased operational pressure.
- **Rationale:** The index rises from 121 to 129 because the Event directly increases the intensity and geographic breadth of kinetic activity across the existing Gulf, Lebanon, Iraq and Yemen theaters and sustains confrontation around tanker transit. The increase is large because both attack tempo and dispersed operational activity expand, while no entirely new theater is established.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.
  - The United States and regional partners are conducting coordinated military operations against Iranian naval, missile and coastal capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.

## Event E3

- **Event ID:** E3
- **Actor:** US
- **Action:** Expand the coordinated military campaign by intensifying strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities, while tightening enforcement of the naval blockade and increasing escorted shipping operations through the Strait of Hormuz.
- **Rationale:** Iran's expanded attacks directly threaten US forces, regional partners and freedom of navigation. A more forceful response could impose additional costs, restore deterrence, protect maritime operations and demonstrate that continued coercion will not compel US withdrawal.
- **Selection Weight:** 0.8
- **Source WorldState:** W3
- **Resulting WorldState:** W4

### Resulting WorldState: W4

#### WorldVariable: Oil Price

- **Variable Name:** Oil Price
- **Index:** 117
- **Change from W3:** 116 → 117
- **Status:** Brent prices remain elevated as the United States intensifies strikes on Iranian capabilities and tightens enforcement of the Strait of Hormuz blockade. Increased escorted shipping provides some countervailing support for transit, but expanded military operations sustain a substantial disruption and security risk premium.
- **Rationale:** The index rises from 116 to 117 because intensified US strikes and tighter blockade enforcement increase immediate regional escalation and disruption risks. The increase is limited because expanded escorted shipping operations provide some reassurance regarding tanker movement and partially offset the added risk.
- **Conditions:**
  - Middle Eastern oil production and exports remain materially below pre-conflict levels.
  - Global inventories have undergone substantial cumulative draws, limiting available market buffers.
  - Shipping disruptions in the Strait of Hormuz and Bab el-Mandeb continue to create price volatility.
  - Iran has declared the Strait of Hormuz closed to commercial shipping and is using force to prevent tanker transit.
  - The United States and regional partners are conducting military operations and escorted shipping to reduce threats to tanker transit.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and regional partner military assets.
  - The United States has intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities and tightened blockade enforcement.

#### WorldVariable: Hormuz Oil Flow

- **Variable Name:** Hormuz Oil Flow
- **Index:** 69
- **Change from W3:** 65 → 69
- **Status:** Oil transit through the Strait of Hormuz increases modestly as the United States expands escorted shipping operations, although the enforced closure, tighter blockade and intensified military activity continue to constrain reliability and volume.
- **Rationale:** The index rises from 65 to 69 because increased escorted shipping directly enables more tanker movement through the Strait despite continuing Iranian attacks and the declared closure. The increase is limited because tighter blockade enforcement and intensified strikes maintain a high-risk operating environment and do not restore normal transit.
- **Conditions:**
  - Iranian threats and attacks on commercial shipping continue to restrict transit.
  - Iran has formally declared the Strait of Hormuz closed to commercial shipping and ordered its forces to prevent tanker transit.
  - A US naval blockade of Iranian ports remains in place according to the latest supplied conflict assessments.
  - Iranian attacks on US naval forces operating in the Gulf increase the immediate security risk to maritime transit.
  - Alternative pipelines, overland routes, ship-to-ship transfers and military escorts are partially sustaining oil movement.
  - The United States and regional partners have expanded escorted shipping operations through the Strait of Hormuz.
  - US and regional military operations are targeting Iranian naval, missile and coastal capabilities that threaten tanker transit.
  - Iran has intensified dispersed missile, drone and asymmetric maritime attacks against US and partner military assets while maintaining the enforced closure.
  - The United States has tightened blockade enforcement and increased escorted shipping operations through the Strait.

#### WorldVariable: Regional Diplomatic Engagement

- **Variable Name:** Regional Diplomatic Engagement
- **Index:** 78
- **Change from W3:** 81 → 78
- **Status:** Diplomatic engagement weakens as the United States expands military strikes and blockade enforcement while Iran continues attacks and the enforced Strait of Hormuz closure. Existing mediation and indirect contacts remain possible but are further obstructed by active escalation.
- **Rationale:** The index falls from 81 to 78 because the Event expands direct US military action and tightens coercive measures while existing contacts are already stalled. The decline is moderate because the current state still records possible indirect contacts and regional mediation, but their immediate operating space is reduced.
- **Conditions:**
  - The June 2026 memorandum of understanding has expired and its commitments are not being implemented.
  - Indirect US-Iran contacts and regional mediation remain possible but are currently stalled.
  - The status of any Iranian willingness to restore IAEA access remains disputed.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces further obstruct immediate de-escalation.
  - The United States and regional partners are conducting an expanded coordinated military campaign against Iranian capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks across existing theaters and continues enforcing the Strait of Hormuz closure.
  - The United States has intensified strikes and tightened blockade enforcement, further narrowing near-term de-escalation space.

#### WorldVariable: Economic Pressure on Iran

- **Variable Name:** Economic Pressure on Iran
- **Index:** 110
- **Change from W3:** 106 → 110
- **Status:** Economic pressure on Iran increases as the United States tightens enforcement of the naval blockade and intensifies strikes on capabilities relevant to maritime access and exports. No additional sanctions or financial restrictions are specified.
- **Rationale:** The index rises from 106 to 110 because tighter blockade enforcement directly increases constraints on Iranian maritime access and export-related activity, while strikes on coastal and naval capabilities reinforce those constraints. The increase is moderate because the Event does not add new sanctions, trade restrictions or financial measures.
- **Conditions:**
  - The US naval blockade of Iranian ports continues.
  - Iranian oil exports and access to associated financial and shipping services remain heavily restricted.
  - The US has announced an expanded sanctions campaign targeting additional sectors and sanctions-evasion networks.
  - US and regional military operations are further constraining Iranian naval, missile and coastal capabilities relevant to maritime access and oil exports.
  - Iran's intensified attacks and enforced Strait of Hormuz closure are increasing the operational constraints on Iranian maritime and export activity.
  - The United States has tightened enforcement of the naval blockade and intensified strikes on Iranian capabilities supporting maritime and export activity.

#### WorldVariable: US/Regional Partner Cohesion

- **Variable Name:** US/Regional Partner Cohesion
- **Index:** 118
- **Change from W3:** 115 → 118
- **Status:** US and regional partner cohesion increases as the coordinated campaign expands and escorted shipping operations are increased in response to the shared threat from Iran. Existing partner concerns about escalation and conflict costs remain.
- **Rationale:** The index rises from 115 to 118 because the Event expands coordinated military operations and maritime protection involving the United States and regional partners. The increase is moderate rather than large because existing conditions already record active cooperation and continuing partner concerns about escalation.
- **Conditions:**
  - US, Israeli and selected Gulf security cooperation remains active during the conflict.
  - Regional partners face continuing Iranian or Iran-linked attacks on shipping, energy infrastructure and territory.
  - Gulf states are pursuing or considering additional regional security arrangements alongside their US relationships.
  - Iran's declared closure of the Strait of Hormuz and attacks on US naval forces increase the shared security threat to the United States and regional partners.
  - The United States and regional partners are conducting a coordinated military campaign and expanded escorted shipping operations.
  - Iran has expanded attacks against US and partner military assets and supports Iran-linked operations across existing theaters.
  - The United States has intensified the coordinated campaign and increased escorted shipping operations with regional partners.

#### WorldVariable: Iran Domestic Stability

- **Variable Name:** Iran Domestic Stability
- **Index:** 88
- **Change from W3:** 92 → 88
- **Status:** Iran's governing system remains in power but faces greater military, economic and organizational strain as US strikes intensify and blockade enforcement tightens. Security-force capacity remains present, and no domestic political collapse is established.
- **Rationale:** The index falls from 92 to 88 because intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities and tighter blockade enforcement increase pressure on the state. The decline is substantial but limited because the current state still records regime control and continuing security-force capacity without a distinct domestic breakdown.
- **Conditions:**
  - Mojtaba Khamenei remains the selected Supreme Leader but has not appeared publicly.
  - Iranian security forces continue arrests, repression and execution of people accused of protest or national-security offenses.
  - High inflation, currency deterioration, energy shortages and reduced oil revenue continue to burden households and the state.
  - US and regional military operations are imposing additional pressure on Iranian naval, missile and coastal capabilities.
  - Iran's expanded regional coercion campaign increases military, economic and organizational strain on the state.
  - The United States has intensified strikes on Iranian command-and-control and military capabilities and tightened blockade enforcement, increasing pressure on state operations.

#### WorldVariable: US-Iran Tension

- **Variable Name:** US-Iran Tension
- **Index:** 134
- **Change from W3:** 128 → 134
- **Status:** US-Iran tension rises sharply as the United States intensifies strikes on Iranian military and command capabilities, tightens the naval blockade and expands escorted shipping while Iran maintains attacks and the enforced Strait of Hormuz closure.
- **Rationale:** The index rises from 128 to 134 because the Event directly expands US military action, blockade enforcement and the maritime confrontation with Iran. The increase is substantial given the escalation across several operational domains, while it remains below an extreme ceiling because the Event continues the existing conflict rather than establishing a wholly new theater.
- **Conditions:**
  - The United States and Iran remain engaged in intermittent military confrontation.
  - US sanctions and the naval blockade remain active while Iran continues to retain asymmetric maritime and strike capabilities.
  - The parties remain divided over sanctions relief, nuclear monitoring and control or administration of the Strait of Hormuz.
  - Iran has declared the Strait of Hormuz closed and launched attacks against US naval forces operating in the Gulf.
  - The United States has launched a coordinated military campaign with regional partners against Iranian naval, missile and coastal capabilities.
  - Iran has intensified missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.
  - The United States has intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities while tightening blockade enforcement.

#### WorldVariable: Regional Armed Conflict

- **Variable Name:** Regional Armed Conflict
- **Index:** 135
- **Change from W3:** 129 → 135
- **Status:** Regional armed conflict intensifies as the United States expands strikes across Iranian military and command capabilities, tightens the blockade and increases escorted shipping operations while Iran continues attacks across existing theaters and maintains the enforced Hormuz closure.
- **Rationale:** The index rises from 129 to 135 because the Event increases the intensity of direct kinetic operations against Iranian missile, drone, naval, coastal and command-and-control capabilities and sustains combat around tanker transit. The increase is substantial because the operational tempo expands across multiple existing conflict domains, without establishing an entirely new theater.
- **Conditions:**
  - Intermittent US-Iran military strikes continue after the collapse of the June memorandum.
  - Iran-linked groups remain active in Lebanon, Iraq and Yemen, with differing levels of capability and territorial control.
  - Attacks and counterattacks continue to affect commercial shipping, Gulf energy facilities and regional military installations.
  - Iran has launched attacks against US naval forces operating in the Gulf and ordered its forces to prevent tanker transit through the Strait of Hormuz.
  - The United States and regional partners are conducting coordinated military operations against Iranian naval, missile and coastal capabilities.
  - Iran has expanded missile, drone and asymmetric maritime attacks against US and partner military assets and supports Iran-linked operations across existing theaters.
  - The United States has intensified strikes on Iranian missile, drone, naval, coastal and command-and-control capabilities and increased military activity around the Strait.

#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)